Paths (local Mac / VSCode)

In [ ]:
import os
from pathlib import Path

DEV_CSV   = "dev.csv"
TRIAL_CSV = "NLI_trial.csv"   # change later to "test.csv"
OUT_DIR   = "outputs"
PRED_OUT  = os.path.join(OUT_DIR, "predictions.csv")

Path(OUT_DIR).mkdir(parents=True, exist_ok=True)

print("DEV_CSV:", DEV_CSV)
print("TRIAL_CSV:", TRIAL_CSV)
print("OUT_DIR:", OUT_DIR)
print("PRED_OUT:", PRED_OUT)

Imports

In [ ]:
import json
import pickle
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, classification_report
from tensorflow.keras.preprocessing.sequence import pad_sequences

Load config.json (fix MAX_LEN mismatch)

In [ ]:
cfg_path = os.path.join(OUT_DIR, "config.json")
cfg = json.load(open(cfg_path))

MAX_LEN = int(cfg["MAX_LEN"])
VOCAB_SIZE = int(cfg["VOCAB_SIZE"])

print("✅ Loaded config:", cfg_path)
print("MAX_LEN:", MAX_LEN, "| VOCAB_SIZE:", VOCAB_SIZE)

Load tokenizer.pkl

In [ ]:
tok_path = os.path.join(OUT_DIR, "tokenizer.pkl")
with open(tok_path, "rb") as f:
    tokenizer = pickle.load(f)

print("✅ Loaded tokenizer:", tok_path)

Tokenise and pad

In [ ]:
def tok_pad(series):
    seqs = tokenizer.texts_to_sequences(series.astype(str).tolist())
    return pad_sequences(seqs, maxlen=MAX_LEN, padding="post", truncating="post")

Load Model

In [ ]:
model_path = os.path.join(OUT_DIR, "best_model.keras")
model = tf.keras.models.load_model(model_path, compile=False)

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

print("✅ Loaded model:", model_path)

Evaluate on dev.csv (has labels)

In [ ]:
dev_df = pd.read_csv(DEV_CSV)
assert {"premise","hypothesis","label"}.issubset(dev_df.columns)

X_dev_p = tok_pad(dev_df["premise"])
X_dev_h = tok_pad(dev_df["hypothesis"])
y_dev   = dev_df["label"].astype(int).values

dev_probs = model.predict([X_dev_p, X_dev_h], batch_size=256, verbose=0).ravel()
dev_pred  = (dev_probs >= 0.5).astype(int)

acc = accuracy_score(y_dev, dev_pred)
p   = precision_score(y_dev, dev_pred, zero_division=0)
r   = recall_score(y_dev, dev_pred, zero_division=0)
f1  = f1_score(y_dev, dev_pred, zero_division=0)

print(f"DEV Accuracy : {acc:.4f}")
print(f"DEV Precision: {p:.4f}")
print(f"DEV Recall   : {r:.4f}")
print(f"DEV F1       : {f1:.4f}\n")
print(classification_report(y_dev, dev_pred, digits=4))

Create predictions.csv for trial/test (NO labels)

In [ ]:
test_df = pd.read_csv(TRIAL_CSV)
assert {"premise","hypothesis"}.issubset(test_df.columns)

X_t_p = tok_pad(test_df["premise"])
X_t_h = tok_pad(test_df["hypothesis"])

probs = model.predict([X_t_p, X_t_h], batch_size=256, verbose=0).ravel()
pred  = (probs >= 0.5).astype(int)

pd.DataFrame({"prediction": pred}).to_csv(PRED_OUT, index=False)
print("✅ Saved:", PRED_OUT)